In [2]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from scipy.linalg import inv

# Kalman Filter Object

In [3]:
class KalmanFilter(object):
    def __init__(self, F, Q, H, R, u):
        """
        Initialize the dynamical system models.

        Parameters
        ----------
        F : ndarray of shape (n,n)
            The state transition model.
        Q : ndarray of shape (n,n)
            The covariance matrix for the state noise.
        H : ndarray of shape (m,n)
            The observation model.
        R : ndarray of shape (m,m)
            The covariance matrix for observation noise.
        u : ndarray of shape (n,)
            The control vector.
        """
        # Initialize given parameters
        self.F = F
        self.Q = Q
        self.H = H
        self.R = R
        self.u = u
   

    def estimate(self, x, P, z):
        """
        Compute the state estimates using the Kalman filter.
        If x and P correspond to time step k, then z is a sequence of
        observations starting at time step k+1.

        Parameters
        ----------
        x : ndarray of shape (n,)
            The initial state estimate.
        P : ndarray of shape (n,n)
            The initial error covariance matrix.
        z : ndarray of shape (m,N)
            Sequence of N observations (each column is an observation).

        Returns
        -------
        out : ndarray of shape (n,N)
            Sequence of state estimates (each column is an estimate).
        Ps : ndarray of shape (n,n,N)
            Sequence of smoothed covariance estimates.
        """
        n = len(x)
        N = z.shape[1]

        # Store initial state estimates
        out = np.zeros((n, N))
        Ps = np.zeros((n, n, N))
        Ps_predicted = np.zeros((n, n, N))
        x0, P0 = x, P
        out[:, 0] = x0
        Ps[:, :, 0] = P0

        for i in range(1, N):
            # Predict steps for xk and Pk
            xk = self.F @ x0 + self.u
            Pk = self.F @ P0 @ self.F.T + self.Q
            Pk = (Pk + Pk.T) / 2
            Ps_predicted[:, :, i] = Pk

            # Update steps for Kk, xk, and Pk
            Kk = np.linalg.solve((self.H @ Pk @ self.H.T + self.R).T,
                                 (Pk @ self.H.T).T).T
            xk = xk + Kk @ (z[:, i] - self.H @ xk)
            Pk = (np.eye(n) - Kk @ self.H) @ Pk
            Pk = (Pk + Pk.T) / 2

            out[:, i] = xk
            Ps[:, :, i] = Pk
            x0, P0 = xk, Pk

        return out, Ps, Ps_predicted
            
    
    def predict(self, x, k):
        """
        Predict the next k states in the absence of observations.

        Parameters
        ----------
        x : ndarray of shape (n,)
            The current state estimate.
        k : integer
            The number of states to predict.

        Returns
        -------
        out : ndarray of shape (n,k)
            The next k predicted states.
        """
        # Initialize array to store predicted states
        n = len(x)
        out = np.zeros((n, k))
        x0 = x
        out[:, 0] = x0

        for i in range(1, k):
            # Perform update step without observations
            xk = np.dot(self.F, x0) + self.u
            out[:, i] = xk
            x0 = xk

        return out
    

    def rewind(self, x, P, Ps_filtered, Ps_predicted):
        """
        Predict the k states and smoothed covariances
        preceding the current state estimate x.

        Parameters
        ----------
        x : ndarray of shape (n,)
            The current state estimate.
        P : ndarray of shape (n,n)
            The current smoothed covariance estimate.
        T : integer
            The number of preceding states to predict.

        Returns
        -------
        out : ndarray of shape (n,k)
            The k preceding predicted states.
        Ps : ndarray of shape (n,n,k)
            The k preceding smoothed covariances.
        """
        # Initialize prediction arrays
        T = Ps_filtered.shape[2]
        n = len(x)
        out = np.zeros((n, T))
        Ps = np.zeros((n, n, T))

        # Store known state as last column
        out[:, -1] = x
        Ps[:, :, -1] = P
        
        for i in range(T-2, -1, -1):
            # Predicted covariance one step aghead
            P_filt = Ps_filtered[:, :, i]
            P_pred = Ps_predicted[:, :, i+1]
            P_pred = (P_pred + P_pred.T) / 2

            # RTS Smoother gain
            Gk = np.linalg.solve(P_pred.T, (P_filt @ self.F.T).T).T

            out[:, i] = out[:, i+1]
            out[:, i] = Ps_filtered[:, :, i] @ np.zeros(n)

            x_filt = out[:, i+1]
            Pk = P_filt + Gk @ (Ps[:, :, i+1] - P_pred) @ Gk.T
            Ps[:, :, i] = (Pk + Pk.T) / 2

        return out, Ps
    
    def smooth(self, x, P, z):
        """Combines forward and backward pass."""
        n = len(x)
        T = z.shape[1]

        # Forward pass
        xs = np.zeros((n, T))
        Ps_filt = np.zeros((n, n, T))
        Ps_pred = np.zeros((n, n, T))

        x0, P0 = x, P
        xs[:, 0] = x0
        Ps_filt[:, :, 0] = P0
        Ps_pred[:, :, 0] = P0

        for i in range(1, T):
            xk = self.F @ x0 + self.u
            Pk = self.F @ P0 @ self.F.T + self.Q
            Pk = (Pk + Pk.T) / 2
            Ps_pred[:, :, i] = Pk

            Kk = np.linalg.solve((self.H @ Pk @ self.H.T + self.R).T,
                                (Pk @ self.H.T).T).T
            xk = xk + Kk @ (z[:, i] - self.H @ xk)
            Pk = (np.eye(n) - Kk @ self.H) @ Pk
            Pk = (Pk + Pk.T) / 2

            xs[:, i] = xk
            Ps_filt[:, :, i] = Pk
            x0, P0 = xk, Pk

        # Backward pass
        x_smooth = xs.copy()
        Ps_smooth = Ps_filt.copy()

        for i in range(T-2, -1, -1):
            P_pred = Ps_pred[:, :, i+1]
            P_pred = (P_pred + P_pred.T) / 2

            Gk = np.linalg.solve(P_pred.T, (Ps_filt[:, :, i] @ self.F.T).T).T

            x_smooth[:, i] = xs[:, i] + Gk @ (x_smooth[:, i+1] - self.F @ xs[:, i] - self.u)
            Pk = Ps_filt[:, :, i] + Gk @ (Ps_smooth[:, :, i+1] - P_pred) @ Gk.T
            Ps_smooth[:, :, i] = (Pk + Pk.T) / 2

        return x_smooth, Ps_smooth, Ps_filt

# Known parameters for Kalman Filter

In [4]:
# State transiiton matrix 
# Follows forward Euler x_{k+1} = x_k + dx
F = np.array([
    [1, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 0, 0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0, 0, 1, 0],
    [0, 0, 0, 1, 0, 0, 0, 1],
    [0, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 1]
])

# First four states are observed at each step
H = np.array([
    [1, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0]
])

n = F.shape[0]  # Number of predicted states
m = H.shape[0]  # Number of hidden/observed states

# We don't use control for this
u = np.zeros(n)

# Create test/training splits from category 3 data

In [8]:
# Category 3 videos include people moving around in the screen
video_files = ['410', '411', '516', '517', '526', '528', '529', '530', '531', '533', 
            #    '557', ''
               '558', '559', '562']

X_train, X_test = train_test_split(video_files, test_size=0.3, random_state=42)

# Helper functions for EM estimation

In [9]:
def get_bounding_box(filename):
    with open(filename) as f:
        data = f.readlines()

    combined = ' '.join(data)
    start = combined.index("{") + 3
    end = combined.index("}")
    points = [coord.split(" ") for coord in combined[start:end].strip().split("\n ")]
    points = np.array(points).astype(float)

    max_coords = np.max(points, axis=0)
    min_coords = np.min(points, axis=0)
    x, y = min_coords
    w, h = max_coords - min_coords
    
    return x, y, w, h

def get_observations(video_id):
    filename = f"data/{video_id}/annot"
    frames = sorted([file for file in os.listdir(filename) if file.endswith(".pts")])
    Z = []

    for frame in frames:
        file = filename + "/" + frame
        x, y, w, h = get_bounding_box(file)
        Z.append([x, y, w, h])

    return np.array(Z).T

# EM-estimation using guesses for Q and R

In [10]:
# Initial guesses for noise covariances
Q = 0.1 * np.eye(n)
R = 500 * np.eye(m)

n_em_iters = 20

for em_iter in range(n_em_iters):
    # Instantiate a Kalman filter
    kf = KalmanFilter(F, Q, H, R, u)

    # Accumulated estimates across all training videos
    R_accum = np.zeros((m, m))
    Q_accum = np.zeros((n, n))
    total_frames = 0

    for video_id in X_train:
        Z = get_observations(video_id)
        T = Z.shape[1]

        # Expectation step: Forward Kalman filter
        z0 = Z[:, 0] 
        x0 = np.concatenate((z0, np.zeros(m)))  # Guess zero initial velocity
        P0 = np.diag([100, 100, 100, 100, 
                      10, 10, 10, 10]).astype(float)
        
        x_smooth, Ps_smooth, Ps_filt = kf.smooth(x0, P0, Z)

        residuals = Z - H @ x_smooth
        if np.any(np.diag(R_accum) > 1e8):
            print(f"R blowing up on video {video_id}, skipping video")
            continue

        # Maximization step: Accumulate R and Q updates
        for i in range(T):
            residual = residuals[:, i]
            R_accum += np.outer(residual, residual) + H @ Ps_smooth[:, :, i] @ H.T

        for i in range(1, T):
            P_pred = F @ Ps_smooth[:, :, i-1] @ F.T + Q
            P_pred = (P_pred + P_pred.T) / 2
            Gk = np.linalg.solve(P_pred.T, (Ps_smooth[:, :, i-1] @ F.T).T).T
            P_cross = Ps_smooth[:, :, i] @ Gk.T

            Q_accum += (Ps_smooth[:, :, i]
                        - F @ P_cross.T
                        - P_cross @ F.T
                        + F @ Ps_smooth[:, :, i-1] @ F.T)
            
        total_frames += T

    # Noramlize by total frames across all training videos
    R = R_accum / total_frames
    R = (R + R.T) / 2
    R = np.maximum(R, 1e-4 * np.eye(m))

    Q = Q_accum / total_frames
    Q = (Q + Q.T) / 2
    Q = np.maximum(Q, 1e-4 * np.eye(n))

    print(f"EM iter {em_iter+1}/{n_em_iters} complete - "
          f"diag(R): {np.diag(R).round(2)}, diag(Q): {np.diag(Q).round(4)}")

EM iter 1/20 complete - diag(R): [50.5  34.94 25.17 25.39], diag(Q): [0.2041 0.2041 0.2041 0.2041 0.1509 0.1509 0.1509 0.1509]
EM iter 2/20 complete - diag(R): [7.99 6.29 3.87 4.37], diag(Q): [0.5425 0.4766 0.4215 0.423  0.1824 0.1754 0.1691 0.1693]
EM iter 3/20 complete - diag(R): [2.25 2.16 1.47 1.86], diag(Q): [1.00000e-04 1.00000e-04 1.03427e+01 1.00000e-04 1.53800e-01 1.42000e-01
 1.54600e-01 1.24700e-01]
EM iter 4/20 complete - diag(R): [1.17 1.42 1.2  1.45], diag(Q): [3.1130e-01 3.5770e-01 2.2823e+00 1.0000e-04 4.9000e-02 6.1300e-02
 3.4600e-02 1.5700e-01]
EM iter 5/20 complete - diag(R): [1.05 1.15 0.86 1.17], diag(Q): [1.6010e-01 3.6020e-01 1.1207e+00 2.0000e-04 4.4100e-02 1.2300e-02
 1.7100e-02 1.2910e-01]
EM iter 6/20 complete - diag(R): [1.   1.32 0.68 1.13], diag(Q): [0.2227 0.1741 0.6563 0.0022 0.014  0.0623 0.0102 0.1062]
EM iter 7/20 complete - diag(R): [1.2  1.06 0.62 1.18], diag(Q): [1.935e-01 2.533e-01 4.367e-01 1.000e-04 1.110e-02 6.260e-02 7.000e-03
 8.780e-02]
EM 